# Узел Б — обучение модели и Flask-сервер

Ноутбук запускает контейнер `ml-trainer`. Flask слушает порт 5000. После передачи `clean_data.csv` с узла А endpoint `GET /predict` обучает `DecisionTreeClassifier` на 80% размеченных строк и возвращает прогнозы для оставшихся 20%.

Заполните `node_b/.env`. На узле А укажите абсолютный путь `node_b/data` в `NODE_B_DATA_DIR`.

## 1. Найти папку узла Б и запустить сервер

In [ ]:
from pathlib import Path
import json
import subprocess
import time
from urllib.request import urlopen

current = Path.cwd().resolve()
NODE_B_DIR = current if (current / 'compose.yaml').is_file() and (current / 'ml_service.py').is_file() else current / 'node_b'
assert (NODE_B_DIR / 'compose.yaml').is_file(), f'Не найдена папка node_b рядом с {current}'
assert (NODE_B_DIR / '.env').is_file(), 'Скопируйте node_b/.env.example в node_b/.env.'
subprocess.run(['docker', 'compose', 'up', '--build', '-d'], cwd=NODE_B_DIR, check=True)
subprocess.run(['docker', 'ps', '--filter', 'name=ml-trainer'], check=True)

## 2. Проверить, что Flask слушает порт

Сначала сервер запущен, но CSV может ещё не прийти. `data_ready: false` в таком случае нормально.

In [ ]:
with urlopen('http://127.0.0.1:5000/health', timeout=5) as response:
    health = json.loads(response.read().decode('utf-8'))
print(health)

## 3. Получить прогнозы после передачи файла с узла А

Сначала выполните ноутбук узла А. Он отправит CSV и вызовет тот же endpoint. Эту ячейку можно запустить повторно, чтобы просмотреть JSON-ответ на узле Б.

In [ ]:
try:
    with urlopen('http://127.0.0.1:5000/predict', timeout=30) as response:
        prediction_result = json.loads(response.read().decode('utf-8'))
    print(json.dumps(prediction_result, ensure_ascii=False, indent=2))
except Exception as error:
    print(f'Прогноз пока недоступен: {error}')
    print('Проверьте, что узел А уже передал clean_data.csv и что контейнер ml-trainer запущен.')

`GET /health` проверяет сервер и наличие CSV. `GET /predict` запускает обучение и возвращает JSON с точностью и прогнозами. Контейнер публикует порт `5000:5000`; фильтр DOCKER-USER на хосте Б ограничивает доступ адресом узла А.